In [ ]:
!pip install -U bitsandbytes

In [ ]:
import wandb
from huggingface_hub import login

wandb.login(key='wand_key')
login(token='hf_token')

In [ ]:
%%writefile train_ddp.py

import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer
import wandb
from huggingface_hub import login

# Hugging Face & W&B login
login(token="hf_token")
wandb.login(key='wandb_token')

local_rank = int(os.environ.get("LOCAL_RANK", 0))
torch.cuda.set_device(local_rank)

# ============================================================================
# CONFIG CLASS WITH OPTUNA BEST PARAMETERS
# ============================================================================
class Config:
    train_file = "/kaggle/input/datasets/pradippokhrel77/flytech-and-alpaca-datasets-for-coding-problem/refined_train.jsonl"
    output_dir = "/kaggle/working/"
    model_name = "meta-llama/Llama-2-7b-chat-hf"

    num_epochs = 2
    per_device_train_batch_size = 4  # Optuna
    gradient_accumulation_steps = 4
    learning_rate = 2.9089891826881823e-05  # Optuna
    max_seq_length = 512

    lora_r = 8  # Optuna
    lora_alpha = 32  # Optuna
    lora_dropout = 0.096107628615683  # Optuna

    seed = 42
    logging_steps = 10
    save_strategy = "steps"
    save_steps = 50
    eval_strategy = "steps"
    eval_steps = 50

    wandb_project = "Final_finetuning"
    wandb_run_name = "llama2-7b-chat-qlora-masked"

    batch_size = 4  # Optuna
    warmup_ratio = 0.06934415060563733  # Optuna
    lr_scheduler = "cosine"  # Optuna

config = Config()

torch.manual_seed(config.seed)
torch.cuda.manual_seed_all(config.seed)

# ============================================================================
# PREPROCESS FUNCTION WITH INSTRUCTION MASKING
# ============================================================================
def preprocess_function_with_masking(examples, tokenizer, max_seq_length):
    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    prompts = examples["prompt"]
    completions = examples["code"]

    for prompt, completion in zip(prompts, completions):
        if not prompt.endswith((' ', '\n', '\t')) and not completion.startswith((' ', '\n', '\t')):
            full_text = prompt + ' ' + completion
        else:
            full_text = prompt + completion

        full_text += tokenizer.eos_token

        tokenized_full = tokenizer(full_text, max_length=max_seq_length, truncation=True, padding=False)
        tokenized_prompt = tokenizer(prompt, max_length=max_seq_length, truncation=True, padding=False)
        prompt_length = len(tokenized_prompt["input_ids"])

        input_ids = tokenized_full["input_ids"]
        labels = [-100] * prompt_length + input_ids[prompt_length:]

        input_ids_list.append(input_ids)
        attention_mask_list.append(tokenized_full["attention_mask"])
        labels_list.append(labels)

    return {"input_ids": input_ids_list, "attention_mask": attention_mask_list, "labels": labels_list}

# ============================================================================
# DATA COLLATOR
# ============================================================================
class DataCollatorForCompletionOnlyLM:
    def __init__(self, tokenizer, max_length=None):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        if self.max_length:
            max_len = min(max_len, self.max_length)

        batch = {"input_ids": [], "attention_mask": [], "labels": []}

        for feature in features:
            input_ids = feature["input_ids"][:max_len]
            attention_mask = feature["attention_mask"][:max_len]
            labels = feature["labels"][:max_len]

            padding_length = max_len - len(input_ids)
            input_ids += [self.tokenizer.pad_token_id] * padding_length
            attention_mask += [0] * padding_length
            labels += [-100] * padding_length

            batch["input_ids"].append(input_ids)
            batch["attention_mask"].append(attention_mask)
            batch["labels"].append(labels)

        return {k: torch.tensor(v) for k, v in batch.items()}

# ============================================================================
# LOAD DATASET
# ============================================================================
dataset = load_dataset("json", data_files=config.train_file, split="train")
dataset = dataset.shuffle(seed=config.seed)
val_size = int(len(dataset) * 0.05)
train_size = len(dataset) - val_size
train_dataset = dataset.select(range(train_size))
eval_dataset = dataset.select(range(train_size, train_size + val_size))

# ============================================================================
# TOKENIZER
# ============================================================================
tokenizer = AutoTokenizer.from_pretrained(config.model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

train_dataset = train_dataset.map(lambda x: preprocess_function_with_masking(x, tokenizer, config.max_seq_length), batched=True, remove_columns=dataset.column_names)
eval_dataset = eval_dataset.map(lambda x: preprocess_function_with_masking(x, tokenizer, config.max_seq_length), batched=True, remove_columns=dataset.column_names)

# ============================================================================
# QUANTIZATION CONFIG
# ============================================================================
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

# ============================================================================
# LOAD MODEL
# ============================================================================
model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    quantization_config=bnb_config,
    device_map={"": local_rank},
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

# ============================================================================
# LoRA CONFIG
# ============================================================================
peft_config = LoraConfig(
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# ============================================================================
# COMPUTE WARMUP STEPS FROM RATIO
# ============================================================================
total_training_steps = (len(train_dataset) // (config.batch_size * config.gradient_accumulation_steps)) * config.num_epochs
config.warmup_steps = int(total_training_steps * config.warmup_ratio)

# ============================================================================
# TRAINING ARGUMENTS
# ============================================================================
training_args = TrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.num_epochs,
    per_device_train_batch_size=config.batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    lr_scheduler_type=config.lr_scheduler,
    warmup_steps=config.warmup_steps,
    logging_steps=config.logging_steps,
    save_strategy=config.save_strategy,
    save_steps=config.save_steps,
    save_total_limit=None,
    eval_strategy=config.eval_strategy,
    eval_steps=config.eval_steps,
    push_to_hub=True,
    hub_model_id="pradip777/llama2_finetuning",
    hub_private_repo=True,
    hub_strategy="every_save",
    bf16=False,
    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    report_to="wandb",
    run_name=config.wandb_run_name,
    seed=config.seed,
)

# ============================================================================
# DATA COLLATOR
# ============================================================================
data_collator = DataCollatorForCompletionOnlyLM(tokenizer=tokenizer, max_length=config.max_seq_length)

# ============================================================================
# TRAINER
# ============================================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

# ============================================================================
# START TRAINING
# ============================================================================
trainer.train()

# ============================================================================
# SAVE MODEL
# ============================================================================
trainer.save_model(config.output_dir)
tokenizer.save_pretrained(config.output_dir)

if int(os.environ.get("LOCAL_RANK", 0)) == 0:
    wandb.finish()

In [ ]:
!torchrun --nproc_per_node=2 train_ddp.py